In [2]:
#@title Environent Setup
!pip install --quiet optuna
!pip install --quiet torch

# Connect to drive
from google.colab import drive, userdata
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/ML4Finance/OnionScripts')
import torch
import optuna
import pandas as pd
from OnionNN import OnionNN
import Embedders
import os
import json
import OnionTrainer
import dataProcessor as dP  # Your dataset and dataloader module
from torch.utils.data import DataLoader, random_split
from torch.cuda.amp import autocast, GradScaler
import warnings
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

def smart_load(model: torch.nn.Module, checkpoint_path: str):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')  # or 'cuda' if needed
    model_dict = model.state_dict()
    pretrained_dict = checkpoint

    loaded, not_loaded = [], []

    # Loop through all model parameters
    for k in model_dict:
        if k in pretrained_dict and model_dict[k].shape == pretrained_dict[k].shape:
            model_dict[k] = pretrained_dict[k]
            loaded.append(k)
        else:
            not_loaded.append(k)

    # Load what matches
    model.load_state_dict(model_dict)

    print(f"✅ Loaded: {len(loaded)} layers.")
    print(f"⚠️  Randomly initialized: {len(not_loaded)} layers.")
    for k in not_loaded:
        print(f" - {k}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
#@title Training setup

#—————————————————————————————————————————————————————#
device = 'cuda'
onion_probs_path    = '/content/drive/MyDrive/ML4Finance/onion_probs.parquet'
lstm_dataset_path   = '/content/drive/MyDrive/ML4Finance/lstm_dataset.parquet'
mds_embedding_path  = '/content/drive/MyDrive/ML4Finance/trained_model/mds_embeddings.csv'
lstm_weights_path   = '/content/drive/MyDrive/ML4Finance/trained_model/lstm_weights.pt'
#—————————————————————————————————————————————————————#
# Load data
batch_size     = 256 # @param ["32","64","128","256","512","1024"] {"type":"raw"}
seq_len        = 168 # @param ["168", "336"] {"type":"raw","placeholder":"hours"}
prediction_len = 24
onion_p_df     = pd.read_parquet(onion_probs_path)
#—————————————————————————————————————————————————————#
# Obtain basically everything
processed, scalers, zones, hours, dows, months, feature_names, gas_idx, full_index = OnionTrainer.preprocess_and_scale(lstm_dataset_path)
onion_probs_dict = OnionTrainer.extract_onion_tensor(onion_p_df, zones)

#—————————————————————————————————————————————————————#
# Set input size as prices + number of features
input_size = len(feature_names)
# Create masks to extract data for training and validation
mask_train = (full_index.year <= 2023)
mask_val   = (full_index.year == 2024)
# Extract subsets
processed_train = {zone: processed[zone][mask_train] for zone in zones }
processed_val   = {zone: processed[zone][mask_val]   for zone in zones }
# For training, extract time-related features
hours_train  = hours[mask_train]
dows_train   = dows[mask_train]
months_train = months[mask_train]
# For validation, extract time-related features
hours_val  = hours[mask_val]
dows_val   = dows[mask_val]
months_val = months[mask_val]

#—————————————————————————————————————————————————————#
# Training Dataset
train_dataset = OnionTrainer.OnionTensorDataset(
    scaled_data_dict = processed_train,
    onion_probs_dict = onion_probs_dict,
    hours    = hours_train,
    dows     = dows_train,
    months   = months_train,
    seq_len  = seq_len,
    pred_len = prediction_len,
    gas_idx  = gas_idx
)
# Validation Dataset
val_dataset = OnionTrainer.OnionTensorDataset(
    scaled_data_dict = processed_val,
    onion_probs_dict = onion_probs_dict,
    hours    = hours_val,
    dows     = dows_val,
    months   = months_val,
    seq_len  = seq_len,
    pred_len = prediction_len,
    gas_idx  = gas_idx
)

#—————————————————————————————————————————————————————#
# Training Dataloader
train_dataloader = DataLoader(
    train_dataset,
    batch_size  = batch_size,
    shuffle     = False,
    pin_memory  = True,
    num_workers = 2
)
# Validation Dataloader
val_dataloader = DataLoader(
    val_dataset,
    batch_size  = batch_size,
    shuffle     = False,
    pin_memory  = True,
    num_workers = 2
)

#—————————————————————————————————————————————————————#
# Obtain BZN embeddings

#—————————————————————————————————————————————————————#
# Load pretrained embeddings
pretrained_emb_df = pd.read_csv(mds_embedding_path, index_col=0)
pretrained_bzn = torch.tensor(
    pretrained_emb_df.loc[zones].values,   # reorder
    dtype  = torch.float32,
    device = device
)
# Dimensions
n_bzn, bzn_dim = pretrained_bzn.shape

#—————————————————————————————————————————————————————#
# Get embeddings
ts_emb  = Embedders.TimestampEmbedder(dim_hour = 8, dim_day = 4, dim_month = 4, device = device)
bzn_emb = Embedders.BZNEmbedder(n_bzn = n_bzn, bzn_dim = pretrained_bzn.shape[-1], pretrained = pretrained_bzn, device = device)

#—————————————————————————————————————————————————————#
# Output dimension -> number of hours predicted
output_size = prediction_len


In [1]:
#@title Simple Training
hidden_size   = 256
epochs        = 15 # @param {"type":"integer"}
learning_rate = 0.001 # @param {"type":"number","placeholder":"0.005"}
dropout_rate  = 0.2
num_layers    = 2
device        = 'cuda'

use_pretrained_lstm    = True # @param {"type":"boolean"}
#—————————————————————————————————————————————————————#
# Create model

model   = OnionNN(
    input_size    = input_size,
    hidden_size   = hidden_size,
    output_size   = 24,
    time_embedder = ts_emb,
    bzn_embedder  = bzn_emb,
    num_layers    = num_layers,
    dropout       = dropout_rate,
    device        = device
).to(device)

#—————————————————————————————————————————————————————#
# Load pretrained model
if use_pretrained_lstm:
  with torch.no_grad():
    smart_load(model, lstm_weights_path)

#———— Optimizer
model.set_trainable(time_embed     = False,
                        bzn_embed  = False,
                        lstm       = False,
                        dense      = True) # keep head trainable

#———— Optimizer
optimizer = torch.optim.AdamW(                # ← head–only params
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=learning_rate)
loss_function = torch.nn.MSELoss()

#———— Training
train_history, val_history = OnionTrainer.training_loop(
    model            = model,
    optimizer        = optimizer,
    loss_function    = loss_function,
    train_dataloader = train_dataloader,
    val_loader       = val_dataloader,
    epochs           = epochs,
    device           = device
)

NameError: name 'OnionNN' is not defined

In [ ]:
#@title save params
# Create output directory
output_dir = "drive/MyDrive/ML4Finance/trained_model"
os.makedirs(output_dir, exist_ok = True)
# Save model weights
torch.save(model.state_dict(), os.path.join(output_dir, "OnionNN_weights.pt"))

print("Model and hyperparameters saved.")

In [ ]:
#@title ccac
import pandas as pd
import torch

@torch.no_grad()
def collect_predictions_per_zone(model,
                                 val_loader,
                                 scalers: dict[str, StandardScaler],
                                 idx_to_zone: dict[int, str],
                                 price_idx: int = 0,
                                 device: str = "cuda") -> pd.DataFrame:
    """
    Gather unscaled predictions + targets for every bidding zone.

    Parameters
    ----------
    model        : trained DeepLSTM (already .eval())
    val_loader   : the validation DataLoader
    scalers      : {zone_name: fitted StandardScaler}
    idx_to_zone  : {zone_id (int): zone_name (str)}
    price_idx    : column of the electricity price inside the scaler
    device       : 'cuda' | 'cpu'

    Returns
    -------
    tidy DataFrame with columns:
        zone | horizon | predicted | actual
    """
    rows = []

    for x_seq, z_id, hr_id, dw_id, mo_id, gas_val, y_true in val_loader:
        # move to GPU & swap (T,B,F) -> (T,B,F)
        x_seq  = x_seq.transpose(0, 1).to(device)
        z_id   = z_id.transpose(0, 1).to(device)
        hr_id  = hr_id.transpose(0, 1).to(device)
        dw_id  = dw_id.transpose(0, 1).to(device)
        mo_id  = mo_id.transpose(0, 1).to(device)
        gas_val = gas_val.to(device,    non_blocking = True)                  # (B, 1) – no transpose
        y_true = y_true.to(device)                       # (B, 24)

        y_pred = model(x_seq, z_id, hr_id, dw_id, mo_id, gas_val) # (B, 24)

        # loop over the batch ­– one sample = one sliding window
        for i in range(y_pred.size(0)):
            zone_int   = z_id[0, i].item()               # first timestep still holds zone id
            zone_name  = idx_to_zone[zone_int]
            scaler     = scalers[zone_name]

            # unscale the 24-hour vector
            pred_unscaled = (
                y_pred[i].cpu().numpy() * scaler.scale_[price_idx]
                + scaler.mean_[price_idx]
            )
            true_unscaled = (
                y_true[i].cpu().numpy() * scaler.scale_[price_idx]
                + scaler.mean_[price_idx]
            )

            # store each horizon (0-23 h ahead) separately
            rows.extend(
                {
                    "zone":     zone_name,
                    "horizon":  h,
                    "predicted":p,
                    "actual":   t,
                }
                for h, (p, t) in enumerate(zip(pred_unscaled, true_unscaled))
            )

    return pd.DataFrame(rows)

import os
import matplotlib.pyplot as plt
import pandas as pd    # only for type hints; not required elsewhere

def plot_forecasts_per_zone(
        df: pd.DataFrame,
        days: int,
        out_dir: str = "drive/MyDrive/ML4Finance/trained_model/forecasts"
    ) -> None:
    """
    Save one line-plot per bidding zone (actual vs. predicted).

    Parameters
    ----------
    df      : DataFrame with at least the columns
              ['zone', 'predicted', 'actual'].
              Rows must already be in chronological order
              **within each zone** (the collection routine keeps them so).
    days    : How many recent *days* to show (days × 24 points will be used).
    out_dir : Where the PNG files are written. Created if absent.

    The function names files '<zone>.png', with '/' replaced by '_'.
    """
    os.makedirs(out_dir, exist_ok=True)
    n_pts = days * 24                      # one point per hour

    for zone, g in df.groupby("zone"):
        # keep only the last <days> days
        g = g.tail(n_pts)

        # X-axis: simple hour index; use g.index if it already is a DatetimeIndex
        x = range(len(g))

        plt.figure(figsize=(14, 5))
        plt.plot(x, g["actual"].to_numpy(),
                 label="Actual", linewidth=1.4, linestyle="-", color="blue")
        plt.plot(x, g["predicted"].to_numpy(),
                 label="Predicted", linewidth=1.4, linestyle=":", color="red")

        plt.title(f"Electricity price — {zone} (last {days} days)")
        plt.xlabel("Hour")
        plt.ylabel("€/MWh")
        plt.legend()
        plt.tight_layout()

        fname = zone.replace("/", "_") + ".png"
        plt.savefig(os.path.join(out_dir, fname), dpi=150)
        plt.close()

from torch.utils.data import Subset

def unwrap_dataset(ds):
    """
    Follow .dataset until we reach the underlying (non-Subset) dataset.
    Works even if you have several nested Subset objects.
    """
    while isinstance(ds, Subset):
        ds = ds.dataset
    return ds

# --- build the mapping -------------------------------------------------
base_ds = unwrap_dataset(val_dataset)          # or unwrap_dataset(val_dataloader.dataset)
idx_to_zone = {idx: name for name, idx in base_ds.bzn_to_idx.items()}

df = collect_predictions_per_zone(
        model,               # trained network
        val_dataloader,      # existing loader
        scalers,             # dict from preprocess_and_scale(...)
        idx_to_zone,
        price_idx=0,         # price column inside each scaler
        device=device
)

df.to_csv("drive/MyDrive/ML4Finance/trained_model/pred_vs_true_by_bzn.csv", index=False)
# df is the DataFrame returned by collect_predictions_per_zone(…)


In [ ]:
plot_forecasts_per_zone(df, days=30)

In [5]:
#@title study
import optuna
from copy import deepcopy
use_pretrained_lstm    = True # @param {"type":"boolean"}

def objective(trial):
    # Sample hyperparameters
    lr     = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    epochs = 20

    # Clone embedders to ensure frozen params aren't modified
    ts_emb_copy  = deepcopy(ts_emb)
    bzn_emb_copy = deepcopy(bzn_emb)

    # Init model
    model = OnionNN(
        input_size    = input_size,
        hidden_size   = 256,  # fixed
        output_size   = 24,
        time_embedder = ts_emb_copy,
        bzn_embedder  = bzn_emb_copy,
        num_layers    = 2,
        dropout       = 0.2,
        device        = device
    ).to(device)

    if use_pretrained_lstm:
        with torch.no_grad():
            smart_load(model, lstm_weights_path)

    # Freeze everything but dense
    model.set_trainable(time_embed=False, bzn_embed=False, lstm=False, dense=True)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr
    )

    loss_function = torch.nn.MSELoss()

    train_history, val_history = OnionTrainer.training_loop(
        model, optimizer, loss_function,
        train_dataloader, val_dataloader,
        epochs=epochs, device=device
    )

    final_val_loss = val_history[-1]
    return final_val_loss  # we want to minimize this

def lstm_study(
    *,
    num_trials       : int,
    output_filepath  : str,
    train_dataloader : DataLoader,
    val_dataloader   : DataLoader,
    n_bzn            : int,
    input_size       : int,
    pretrained_embed : torch.nn.Embedding = None,
    device           : str = "cuda"
):
    '''Perform hyper-parameter search and save results.'''

    study = optuna.create_study(direction="minimize")
    study.optimize(lambda trial: objective(trial),
        n_trials          = num_trials,
        show_progress_bar = True,
        gc_after_trial    = True
    )

    # Save study to CSV
    df = study.trials_dataframe()
    df.to_csv(output_filepath, index=False)

    # Print best result
    print(f"\nBest trial:")
    print(f"  Loss   : {study.best_trial.value:.6f}")
    for k, v in study.best_trial.params.items():
        print(f"  {k:<10}: {v}")

lstm_study(
    num_trials       = 30,
    output_filepath  = "optuna_lstm_results.csv",
    train_dataloader = train_dataloader,
    val_dataloader   = val_dataloader,
    n_bzn            = len(zones),
    input_size       = 26,
    pretrained_embed = bzn_emb.embedding,
    device           = 'cuda'
)


[I 2025-06-03 22:53:48,207] A new study created in memory with name: no-name-6710c9eb-8cfc-478f-b7e9-abab1366c3d2


  0%|          | 0/30 [00:00<?, ?it/s]

✅ Loaded: 15 layers.
⚠️  Randomly initialized: 3 layers.
 - fusion.conv.weight
 - fusion.conv.bias
 - dense_out.0.weight
[W 2025-06-03 22:53:48,581] Trial 0 failed with parameters: {'learning_rate': 0.0005325458213021207} because of the following error: RuntimeError('Tensors must have same number of dimensions: got 2 and 3').
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "<ipython-input-5-8c12817a4681>", line 64, in <lambda>
    study.optimize(lambda trial: objective(trial),
                                 ^^^^^^^^^^^^^^^^
  File "<ipython-input-5-8c12817a4681>", line 41, in objective
    train_history, val_history = OnionTrainer.training_loop(
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/ML4Finance/OnionScripts/OnionTrainer.py", line 108, in training_loop
    predictions = mod

RuntimeError: Tensors must have same number of dimensions: got 2 and 3